# P-016 — Main tables (Tables 1–5)

Every number below is read from an aggregate result artifact in `../artifacts/`.
No licensed microdata is used or required — see `../DATA_ACCESS.md`.
Outputs are stored in this notebook, so the tables render on GitHub without running anything.


## Setup


In [ ]:
import json, os
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)

ART = "../artifacts"
def A(name):
    """Load one aggregate result artifact by file stem."""
    with open(os.path.join(ART, name + ".json"), encoding="utf-8") as f:
        return json.load(f)

def show(df, title):
    print(title); print("-" * len(title)); print(df.to_string(index=False)); print()

print("artifacts available:", len(os.listdir(ART)))


artifacts available: 47


## Table 1 — Sample construction and allottee stakes
Panel A is the treatment-universe flow; Panel B the allottee post-money stake distribution.


In [ ]:
u = A("wp13a_universe")
flow = pd.DataFrame(u["flow"])[["n","step","note"]].rename(columns={"n":"N","step":"Step","note":"Definition / use"})
show(flow[["Step","N","Definition / use"]], "Table 1, Panel A. Sample construction")

s = A("wp10ab")["A_stake"]
panelB = pd.DataFrame([
    ("Median", f"{100*s['median']:.2f}%"), ("p25 / p75", f"{100*s['p25']:.1f}% / {100*s['p75']:.1f}%"),
    ("p90", f"{100*s['p90']:.1f}%"), ("Share >= 30%", f"{100*s['ge30']:.1f}%"),
    ("Share >= 50%", f"{100*s['ge50']:.1f}%"), ("n", s["n_stake"]),
], columns=["Statistic","Value"])
show(panelB, "Table 1, Panel B. Allottee post-money stake")


Table 1, Panel A. Sample construction
-------------------------------------
                                                     Step   N                                                                                                                    Definition / use
     Treatment set (first third-party allotment per firm) 393                                                                                        371 paid-in increases · 22 convertible bonds
                With identifiable event date in 2015–2025 360 r1b 353 · document-parsed r1c 7; 22 dated events fall outside the stated sample period (18 before 2015, 4 in 2026) and are excluded
          Event month inside NPS panel (2015-11..2026-05) 321                                                                                                         39 precede/follow the panel
               Window feasible (e−13 ≥ start, e+12 ≤ end) 260                                                                                       

## Table 2 — Announcement cumulative abnormal returns


In [ ]:
w = A("wp8b_car")["windows"]
rows = [(k.replace("m1_p1","[-1,+1]").replace("e0_p1","[0,+1]").replace("e0_p5","[0,+5]").replace("e0_p20","[0,+20]"),
         f"{100*v['mean_CAR']:+.2f}%", f"{v['t']:.2f}", f"[{100*v['ci95'][0]:.1f}%, {100*v['ci95'][1]:.1f}%]",
         f"{100*v['median']:+.1f}%", f"{100*v['pct_pos']:.1f}%") for k,v in w.items()]
show(pd.DataFrame(rows, columns=["Window","Mean CAR","t","95% CI","Median","% positive"]),
     f"Panel A. Equal-weighted proxy (n = {A('wp8b_car')['n_car']})")

b = A("wp10ab")["B_car"]; p = b["purpose_full"]
rows = [("Rescue (working capital / debt repayment)", p["n_surv"], f"{100*p['surv_car11']:+.2f}%", f"{p['surv_bmp_t']:.2f}"),
        ("Growth (facilities / acquisitions)",        p["n_grow"], f"{100*p['grow_car11']:+.2f}%", f"{p['grow_bmp_t']:.2f}"),
        ("Difference", "", f"{100*p['diff']:+.2f} pp", f"Welch p = {p['welch_p']}")]
show(pd.DataFrame(rows, columns=["Purpose","n","Mean CAR [-1,+1]","BMP t"]),
     "Panel B. Purpose split, value-weighted exchange-index proxy")


Panel A. Equal-weighted proxy (n = 317)
---------------------------------------
 Window Mean CAR    t        95% CI Median % positive
[-1,+1]   +3.00% 3.96  [1.5%, 4.5%]  +0.5%      54.9%
 [0,+1]   +1.98% 3.30  [0.8%, 3.2%]  +0.3%      52.7%
 [0,+5]   +3.23% 3.37  [1.4%, 5.1%]  +0.3%      50.8%
[0,+20]   +2.06% 1.48 [-0.6%, 4.8%]  +0.0%      49.8%

Panel B. Purpose split, value-weighted exchange-index proxy
-----------------------------------------------------------
                                  Purpose   n Mean CAR [-1,+1]            BMP t
Rescue (working capital / debt repayment) 174           +5.89%             4.48
       Growth (facilities / acquisitions)  47           +1.59%             1.62
                               Difference             +4.30 pp Welch p = 0.0124


## Table 3 — Average employment path
Listed clean-pool ATT, the pseudo-event gradient, and the two event-year anchors.


In [ ]:
l = A("wp10c_listed"); e = A("wp11e"); fg = A("wp11fg")
rows = [("ATT, avg months +1..+12 (listed clean pool)", l["ATT_avg1_12"]["point"], l["ATT_avg1_12"]["ci"], l["ATT_avg1_12"]["n"]),
        ("ATT, avg months +7..+12", l["ATT_avg7_12"]["point"], l["ATT_avg7_12"]["ci"], l["ATT_avg7_12"]["n"]),
        ("Validated counterfactual model (OOS winner)", e["G4"]["event_effect"], e["G4"]["ci"], e["G4"]["n_event"]),
        ("Trajectory break vs. preceding year", e["G5a_trajbreak"]["tau_accel"], e["G5a_trajbreak"]["ci"], e["G5a_trajbreak"]["n_t"]),
        ("DR-DiD with distress covariates", e["G5b_dr"]["ATT_dr"], e["G5b_dr"]["ci"], e["G5b_dr"]["n_treat"])]
show(pd.DataFrame([(a, f"{b:+.4f}", f"[{c[0]:+.4f}, {c[1]:+.4f}]", d) for a,b,c,d in rows],
                  columns=["Estimate","Point","95% CI","n"]), "Panel A. Benchmarks")

g = fg["g_placebo_grid"]
rows = [(k, f"{v['mean']:+.4f}", f"[{v['mean_ci'][0]:+.4f}, {v['mean_ci'][1]:+.4f}]",
         "yes" if v["mean_equiv"] else "NO", v["n"]) for k,v in g.items()]
rows.append(("event", f"{fg['f_honest']['mean']['effect']:+.4f}",
             f"[{fg['f_honest']['mean']['grid'][0]['ci'][0]:+.4f}, {fg['f_honest']['mean']['grid'][0]['ci'][1]:+.4f}]", "NO", ""))
show(pd.DataFrame(rows, columns=["Date","Mean gap","95% CI",f"Within +/-{fg['delta_mean']}?","n"]),
     "Panel B. Pseudo-event grid (mean)")


Panel A. Benchmarks
-------------------
                                   Estimate   Point             95% CI   n
ATT, avg months +1..+12 (listed clean pool) -0.0563 [-0.0948, -0.0184] 210
                    ATT, avg months +7..+12 -0.0821 [-0.1313, -0.0359] 210
Validated counterfactual model (OOS winner) -0.0954 [-0.1738, -0.0212] 136
        Trajectory break vs. preceding year -0.0787 [-0.1342, -0.0265] 208
            DR-DiD with distress covariates -0.0711 [-0.1223, -0.0222] 210

Panel B. Pseudo-event grid (mean)
---------------------------------
 Date Mean gap             95% CI Within +/-0.0479?   n
 t-36  -0.0058 [-0.0435, +0.0307]               yes 125
 t-30  -0.0115 [-0.0423, +0.0225]               yes 131
 t-24  -0.0520 [-0.0927, -0.0157]                NO 142
 t-18  -0.0576 [-0.1004, -0.0178]                NO 163
event  -0.0809 [-0.1006, -0.0611]                NO    


## Table 4 — The collapse tail


In [ ]:
d = A("wp11d"); c3 = A("wp13c_pooled_placebo")["runs"]
grid = d["grid"]
cur = pd.DataFrame({"c": grid, "Event": d["event"]["point"], "Pseudo (t-24)": d["placebo"]["point"],
                    "Difference": d["ddd"]["point"],
                    "Uniform lo": d["ddd"]["lo_unif"], "Uniform hi": d["ddd"]["hi_unif"]})
cur["Band above 0"] = ["yes" if x > 0 else "no" for x in d["ddd"]["lo_unif"]]
show(cur.round(4), "Panel B. Collapse-probability curve (probability points)")
print("Uniform-significant thresholds:", d["ddd_sig_region_uniform"])
print("Joint max-|t| adjusted p:", d["joint_multiplicity"], "\n")

rows = []
for k, lab in (("A_pooled_cluster","Pooled 4 pseudo-dates, firm-clustered"),
               ("B_t24_cluster","t-24 only, firm-clustered"),
               ("C_common_pooled_cluster","Common sample (event + all 4 dates)")):
    r = c3[k]
    rows.append((lab, r["n_event"], r["n_placebo"],
                 f"{r['p10']['obs']:+.4f}", f"[{r['p10']['ci'][0]:+.4f}, {r['p10']['ci'][1]:+.4f}]",
                 f"{r['mean']['obs']:+.4f}", f"{r['median']['obs']:+.4f}"))
show(pd.DataFrame(rows, columns=["Contrast","n event","n pseudo","p10 diff","95% CI","mean diff","median diff"]),
     "Panel A. Event vs. pseudo-event quantile contrasts")


Panel B. Collapse-probability curve (probability points)
--------------------------------------------------------
    c  Event  Pseudo (t-24)  Difference  Uniform lo  Uniform hi Band above 0
-0.60 0.0457        -0.0002      0.0459      0.0006      0.0912          yes
-0.55 0.0638        -0.0012      0.0650      0.0145      0.1156          yes
-0.50 0.0751        -0.0019      0.0770      0.0219      0.1321          yes
-0.45 0.0719        -0.0045      0.0764      0.0213      0.1315          yes
-0.40 0.0919        -0.0008      0.0927      0.0305      0.1549          yes
-0.35 0.1150        -0.0050      0.1199      0.0515      0.1884          yes
-0.30 0.1249        -0.0055      0.1303      0.0588      0.2019          yes
-0.25 0.1369        -0.0013      0.1382      0.0589      0.2174          yes
-0.20 0.1335        -0.0089      0.1425      0.0592      0.2257          yes
-0.15 0.1639        -0.0143      0.1782      0.0831      0.2733          yes
-0.10 0.1609         0.0085      0.1523

## Table 5 — Recipients vs. non-recipients in the same measured distress state


In [ ]:
b = A("wp12b")["runs"]
def row(key, lab):
    r = b[key]
    return (lab, r.get("n_treated",""), r.get("n_ctrl_events",""),
            f"{r['mean']['obs']:+.4f} [{r['mean']['ci'][0]:+.4f}, {r['mean']['ci'][1]:+.4f}]",
            f"{r['median']['obs']:+.4f} [{r['median']['ci'][0]:+.4f}, {r['median']['ci'][1]:+.4f}]",
            f"{r['p10']['obs']:+.4f} [{r['p10']['ci'][0]:+.4f}, {r['p10']['ci'][1]:+.4f}]")
rows = [row("B_distress_1","Distressed stratum"), row("B_distress_0","Non-distressed stratum")]
p = b["E_pool_distress"]
rows.append(("Pooled (recipient weights)", p["n_treated"], "",
             f"{p['mean']['obs']:+.4f} [{p['mean']['ci'][0]:+.4f}, {p['mean']['ci'][1]:+.4f}]",
             f"{p['median']['obs']:+.4f} [{p['median']['ci'][0]:+.4f}, {p['median']['ci'][1]:+.4f}]",
             f"{p['p10']['obs']:+.4f} [{p['p10']['ci'][0]:+.4f}, {p['p10']['ci'][1]:+.4f}]"))
show(pd.DataFrame(rows, columns=["Stratum","n recipients","n non-recipient events","Mean","Median","p10"]), "Table 5")

d1 = b["B_distress_1"]
print("Distressed stratum, P(outcome <= c):")
print("  recipients        ", d1["cprob_treated"])
print("  weighted non-recip", d1["cprob_ctrl_w"])
print("  thresholds with uniform band above zero:", len(d1["curve"]["sig_region"]), "of", len(d1["curve"]["grid"]))
bal = A("wp12c_balance")["runs"]["B_distress_1"]
print("\nBalance after weighting — by covariate:", bal["smd_by_covariate"])
print("Worst covariate:", bal["worst_covariate"], "| max |SMD| =", bal["max_abs_smd"])
print("Weight diagnostics:", bal["weight_diagnostics"])


Table 5
-------
                   Stratum  n recipients n non-recipient events                       Mean                     Median                        p10
        Distressed stratum           115                  27423 -0.0903 [-0.1675, -0.0122] -0.0015 [-0.0356, +0.0688] -0.3414 [-0.4959, -0.1682]
    Non-distressed stratum            95                  78874 -0.0569 [-0.1065, -0.0094] +0.0186 [-0.0277, +0.0393] -0.2197 [-0.3271, -0.0542]
Pooled (recipient weights)           210                        -0.0752 [-0.1235, -0.0269] +0.0076 [-0.0262, +0.0414] -0.2863 [-0.4054, -0.1673]

Distressed stratum, P(outcome <= c):
  recipients         {'-0.5': 0.1304, '-0.35': 0.1826, '-0.25': 0.2348}
  weighted non-recip {'-0.5': 0.0193, '-0.35': 0.0442, '-0.25': 0.0795}
  thresholds with uniform band above zero: 11 of 11

Balance after weighting — by covariate: {'logsize': -0.036, 'pg': 0.021, 'yr': -0.001, 'lev': 0.01, 'roa': -0.123, 'cash': 0.033, 'imp': -0.004, 'loss': 0.009}
Worst cov